# Loading & Filtering Data

#### INPUT  
- raw_dataset.tsv

#### WHAT THIS NOTEBOOK DOES
- loads the .tsv file downloaded from Swissdox into a dataframe
- filters out all articles that are not in German (language != "de")
- inspects the different rubrics (topic categories) for each source present
- filters out irrelevant topics based on these rubrics or on URLs if no rubric categories are available (SRF, ZWAO)
- filters out duplicate articles
- writes the filtered dataframe into a new .tsv file

#### OUTPUTS 
- filtered_dataset.tsv --> filtered, deduplicated articles


In [ ]:
# imports

from config import load_config as cfg

import pandas as pd
import re

In [2]:
# Load the raw data from the TSV file

df_raw = pd.read_csv(cfg.INPUT_TSV, sep="\t")

print(f"Loaded {len(df_raw)} articles from {cfg.INPUT_TSV}")

df_raw.head()

Loaded 245080 articles from data\raw_dataset.tsv


,id,pubtime,medium_code,medium_name,rubric,regional,doctype,doctype_description,language,char_count,dateline,head,subhead,article_link,content_id,content
0,59217348,2025-12-16 00:00:00+01,NZZ,Neue Zürcher Zeitung,International,NaN,PND,National daily newspaper,de,7943,NaN,Die Rechte erlebt in Chile eine Renaissance,José Antonio Kast hat Bewunderung für Pinochet...,NaN,00001ef5-51e6-8675-aa14-11ff39c435b3,"<tx><au>Alexander Busch, São Paulo</au><p>Der ..."
1,58344855,2025-09-25 00:00:00+02,TA,Tages-Anzeiger,Forum,NaN,PND,National daily newspaper,de,6271,NaN,Briefe an die Redaktion,NaN,NaN,00002088-159f-7651-eae3-9ebc9267047a,<tx><zt>Der Kahlschlag ist unnötig</zt><zt>«Ta...
2,60051005,2026-02-23 00:00:00+01,BZ,Berner Zeitung,Ausland,BZ Ausgabe Stadt + Region Bern,PRD,Regional daily newspaper,de,775,NaN,Bei Trump-Anwesen: Sicherheitskräfte erschiess...,NaN,NaN,000065bc-41f3-9a29-7fb9-dd3b0bb263d7,<tx><p>USA In der Nacht auf gestern ist eine P...
3,60048733,2026-02-23 00:00:00+01,TA,Tages-Anzeiger,International,NaN,PND,National daily newspaper,de,775,NaN,Bei Trump-Anwesen: Sicherheitskräfte erschiess...,NaN,NaN,000065bc-41f3-9a29-7fb9-dd3b0bb263d7,<tx><p>USA In der Nacht auf gestern ist eine P...
4,60050846,2026-02-23 00:00:00+01,BAZ,Basler Zeitung,International,NaN,PRD,Regional daily newspaper,de,775,NaN,Bei Trump-Anwesen: Sicherheitskräfte erschiess...,NaN,NaN,000065bc-41f3-9a29-7fb9-dd3b0bb263d7,<tx><p>USA In der Nacht auf gestern ist eine P...


In [3]:
# use this cell to check the data for a specific source, e.g. to find relevant topic categories for a source or to check the amount of missing values in the rubric column for a source

source_to_check = "ZWAO"

# describe the overall dataset for the source to check, including missing values
df_raw[df_raw["medium_code"]==source_to_check].describe(include='all')


,id,pubtime,medium_code,medium_name,rubric,regional,doctype,doctype_description,language,char_count,dateline,head,subhead,article_link,content_id,content
count,2.552800e+04,25528,25528,25528,0,0,25528,25528,25528,25528.000000,16342,25528,0,25528,25528,25528
unique,NaN,25181,1,1,0,0,1,1,1,NaN,11120,25255,0,25340,25487,25487
top,NaN,2026-02-05 22:06:58+01,ZWAO,20 minuten online,NaN,NaN,WWE,Online medium,de,NaN,Zürich,So hat deine Gemeinde abgestimmt,NaN,https://www.20min.ch/story/heim-em-das-ist-die...,9ca5e272-f1b3-9da1-c9f1-95d02f5672fb,"<tx><ld><p>Jemandem die Liebe gestehen, dem Gr..."
freq,NaN,6,25528,25528,NaN,NaN,25528,25528,25528,NaN,213,3,NaN,3,9,9
mean,5.900482e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3258.257169,NaN,NaN,NaN,NaN,NaN,NaN
std,1.144061e+06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1589.103780,NaN,NaN,NaN,NaN,NaN,NaN
min,5.409531e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,190.000000,NaN,NaN,NaN,NaN,NaN,NaN
25%,5.804409e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2192.000000,NaN,NaN,NaN,NaN,NaN,NaN
50%,5.895619e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2997.000000,NaN,NaN,NaN,NaN,NaN,NaN
75%,6.006723e+07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4027.000000,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# drop all rows which do not have language "de"
print(f"Filtering for German language. Before filtering, there are {len(df_raw)} articles.")

df_raw = df_raw[df_raw["language"] == "de"]

print(f"After filtering for German language, {len(df_raw)} articles remain.")

Filtering for German language. Before filtering, there are 245080 articles.
After filtering for German language, 245080 articles remain.


In [5]:
# list all sources, how many articles they have, and when the earliest and latest article is from each source
print("Sources and article counts:")
print(df_raw["medium_code"].value_counts())
print("\nEarliest and latest article dates per source:")
print(df_raw.groupby("medium_code")["pubtime"].agg(["min", "max"]))

Sources and article counts:
medium_code
BLIO    52129
SRF     33573
AZM     31964
ZWAO    25528
LUZ     20275
SGT     19786
NZZ     15093
TA      14336
BZ      13979
BAZ     13655
WEW      3015
WOZ      1747
Name: count, dtype: int64

Earliest and latest article dates per source:
                                min                     max
medium_code                                                
AZM          2025-06-02 00:00:00+02  2026-05-30 00:00:00+02
BAZ          2025-06-02 00:00:00+02  2026-05-30 00:00:00+02
BLIO         2025-06-01 00:01:14+02  2026-05-31 23:39:18+02
BZ           2025-06-02 00:00:00+02  2026-05-30 00:00:00+02
LUZ          2025-06-02 00:00:00+02  2026-05-30 00:00:00+02
NZZ          2025-06-02 00:00:00+02  2026-05-30 00:00:00+02
SGT          2025-06-02 00:00:00+02  2026-05-30 00:00:00+02
SRF          2025-06-01 02:34:02+02  2026-05-31 23:40:43+02
TA           2025-06-02 00:00:00+02  2026-05-30 00:00:00+02
WEW          2025-06-05 00:00:00+02  2026-05-28 00:00:00+02

In [6]:
# find topic categories per source (used to update the config file with relevant topic categories per source)

# list sources whose rubric values are entirely missing
sources_with_only_missing_rubric = df_raw.groupby("medium_code")["rubric"].apply(lambda values: values.isna().all())
print("Sources with only missing values in the rubric column:")
print(sources_with_only_missing_rubric[sources_with_only_missing_rubric].index.to_numpy())  

#drop all rows with a value in the rubric column that is lower than 0.5 % of articles per source (to get rid of very small topic categories which are not relevant for the analysis)
df_rubric_analysis = df_raw[
    df_raw.groupby(["medium_code", "rubric"])["rubric"].transform("size") >= 0.005 * df_raw.groupby("medium_code")["rubric"].transform("size")
    ].copy()

#print the amount of rubrics per source and the amount of articles per rubric for each source
for source in df_rubric_analysis["medium_code"].unique():
    print(f"\n=== {source} ===")
    #print the amount of rubrics per source
    print(f"Number of rubrics in {source}: {df_rubric_analysis[df_rubric_analysis['medium_code'] == source]['rubric'].nunique()}")
    #print the amount of articles per rubric for each source
    print(df_rubric_analysis[df_rubric_analysis["medium_code"] == source]["rubric"].value_counts())

Sources with only missing values in the rubric column:
['SRF' 'ZWAO']

=== NZZ ===
Number of rubrics in NZZ: 16
rubric
International            2759
Wirtschaft               2142
Meinung und Debatte      1670
Feuilleton               1595
Schweiz                  1544
Zürich und Region        1396
Sport                    1048
Front                     917
Panorama                  786
Forschung und Technik     278
Zürich                    239
Wochenende                159
Mobilität                 105
Schwerpunkt                93
Medien                     86
Literatur und Kunst        86
Name: count, dtype: int64

=== TA ===
Number of rubrics in TA: 10
rubric
Zürich                           2800
Politik & Wirtschaft             2584
International                    2056
Sport                            1735
Kehrseite                        1574
Kultur, Gesellschaft & Wissen    1301
Front                             868
Meinungen                         685
Hintergrund             

In [7]:
# filter out only relevant topic categories per source (as defined in the config file), while keeping all articles from sources without configured relevant rubrics

df_category_filtered = df_raw.copy()

for source in df_category_filtered["medium_code"].unique():
    if source not in cfg.RELEVANT_RUBRICS_PER_SOURCE:
        print(f"Warning: No relevant rubrics defined for source {source} in the config file. Keeping all articles from this source.")
        continue

    relevant_rubrics = cfg.RELEVANT_RUBRICS_PER_SOURCE[source]
    if len(relevant_rubrics) == 0:
        print(f"Warning: Empty list of relevant rubrics defined for source {source} in the config file. Keeping all articles from this source for later manual filtering.")
        continue

    source_mask = df_category_filtered["medium_code"] == source
    keep_mask = df_category_filtered["rubric"].isin(relevant_rubrics)
    df_category_filtered = df_category_filtered[~source_mask | keep_mask]

print(f"After filtering for relevant rubrics per source, {len(df_category_filtered)} articles remain. Amount of articles filtered out: {len(df_raw) - len(df_category_filtered)}")

print("Sources and article counts:")
print(df_category_filtered["medium_code"].value_counts())

After filtering for relevant rubrics per source, 121508 articles remain. Amount of articles filtered out: 123572
Sources and article counts:
medium_code
SRF     33573
ZWAO    25528
BLIO    22015
NZZ      8115
AZM      5721
TA       5325
LUZ      5080
SGT      4928
BAZ      4613
BZ       4089
WEW      1851
WOZ       670
Name: count, dtype: int64


In [8]:
# for source "SRF", list categories from URLs in the /news/ section

srf_mask = df_category_filtered["medium_code"] == "SRF"
srf_news_path = df_category_filtered["article_link"].str.contains(
    r"://(?:www\.)?srf\.ch/news/", regex=True, na=False
)
srf_news_mask = srf_mask & srf_news_path

if srf_news_mask.any():
    # category is the first path segment after /news/
    df_category_filtered.loc[srf_news_mask, "rubric_from_url"] = (
        df_category_filtered.loc[srf_news_mask, "article_link"]
        .str.extract(r"://(?:www\.)?srf\.ch/news/([^/?#]+)", expand=False)
    )

    rubric_counts = df_category_filtered.loc[srf_news_mask, "rubric_from_url"].value_counts(dropna=False)
    min_count = 0.005 * srf_news_mask.sum()
    rubric_counts_filtered = rubric_counts[rubric_counts > min_count]

    print("Topic categories in URL for source SRF (news only, >0.5% of SRF news articles):")
    print(rubric_counts_filtered)
else:
    print("No SRF news articles found in df_category_filtered.")

Topic categories in URL for source SRF (news only, >0.5% of SRF news articles):
rubric_from_url
international     7532
schweiz           6850
wirtschaft        1489
dialog             860
gesellschaft       591
srf-news-chats     112
Name: count, dtype: int64


In [9]:
# keep SRF articles only if their URL-derived rubric is in the configured allowlist
srf_keep_mask = df_category_filtered["rubric_from_url"].isin(cfg.SRF_RELEVANT_RUBRICS_FROM_URL)
non_srf_mask = df_category_filtered["medium_code"] != "SRF"

df_manual_filtered = df_category_filtered[non_srf_mask | srf_keep_mask].copy()

print(
    f"After SRF URL-rubric filtering, {len(df_manual_filtered)} articles remain. "
    f"Amount of SRF articles filtered out: {(~srf_keep_mask & ~non_srf_mask).sum()}"
 )

print("Sources and article counts:")
print(df_manual_filtered["medium_code"].value_counts())

After SRF URL-rubric filtering, 103806 articles remain. Amount of SRF articles filtered out: 17702
Sources and article counts:
medium_code
ZWAO    25528
BLIO    22015
SRF     15871
NZZ      8115
AZM      5721
TA       5325
LUZ      5080
SGT      4928
BAZ      4613
BZ       4089
WEW      1851
WOZ       670
Name: count, dtype: int64


In [10]:
# for source "ZWAO", there are no standardized categories in the URL, so the filtering must be keyword-based

# keep ZWAO articles only if their URL contains at least one configured keep keyword
keep_article_words = [w.lower() for w in cfg.ZWAO_RELEVANT_CONTENT_FROM_URL]
zwao_pattern = "|".join(re.escape(word) for word in keep_article_words)

zwao_keep_mask = df_manual_filtered["article_link"].fillna("").str.lower().str.contains(
    zwao_pattern, regex=True
 )
non_zwao_mask = df_manual_filtered["medium_code"] != "ZWAO"

df_manual_filtered = df_manual_filtered[non_zwao_mask | zwao_keep_mask].copy()

print(
    f"After ZWAO URL keyword filtering, {len(df_manual_filtered)} articles remain. "
    f"Amount of ZWAO articles filtered out: {(~zwao_keep_mask & ~non_zwao_mask).sum()}"
 )
print("Sources and article counts:")
print(df_manual_filtered["medium_code"].value_counts())

After ZWAO URL keyword filtering, 89732 articles remain. Amount of ZWAO articles filtered out: 14074
Sources and article counts:
medium_code
BLIO    22015
SRF     15871
ZWAO    11454
NZZ      8115
AZM      5721
TA       5325
LUZ      5080
SGT      4928
BAZ      4613
BZ       4089
WEW      1851
WOZ       670
Name: count, dtype: int64


In [11]:
# remove duplicate texts (e.g. for articles that are shared across sources from the same publisher)
df_manual_filtered = df_manual_filtered.drop_duplicates(subset=["head"], ignore_index=True)

print(
    f"After deduplication, {len(df_manual_filtered)} articles remain. "
    f"Amount of duplicate articles filtered out: {(df_manual_filtered['head'].duplicated() & ~df_manual_filtered['head'].isna()).sum()}"
 )

After deduplication, 66404 articles remain. Amount of duplicate articles filtered out: 0


In [12]:
# save the filtered dataframe to a new TSV file for further processing

df_manual_filtered.to_csv(cfg.OUTPUT_TSV, sep="\t", index=False)